# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsimaZaheer/Task1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My task is a classification problem because the target is whether a content page is in the declining group or not. I will use a Random Forest classifier because it can combine multiple search, engagement, content-age, and visibility signals and capture non-linear relationships between them. It also provides feature importance that can help explain which observable signals the model relies on. This fits the Refresh / Content Opportunity Scoring lane because the goal is to rank pages for human review rather than guarantee that a refresh will improve performance.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design


I will use a client-grouped train/test split. Pages from the same client should not be placed in both training and test data because pages from one client can share patterns that make the test artificially easy. Grouping by client gives a more honest estimate of how the model performs on clients it did not see during training. The split will be used consistently for both the Random Forest and the baseline comparison.


In [12]:
# Check the number of unique clients
print("Unique clients:", df["client_id"].nunique())

# Show how many pages belong to each client
print("\nPages per client:")
print(df["client_id"].value_counts().head(10))


Unique clients: 32

Pages per client:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [13]:
!git clone https://github.com/AsimaZaheer/Task1.git

fatal: destination path 'Task1' already exists and is not an empty directory.


In [14]:
import pandas as pd

df = pd.read_csv(
    "/content/Task1/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [15]:
import numpy as np
import pandas as pd

# Create the target
# 1 = declining, 0 = not declining
df["target"] = (df["trend_direction"] == "down").astype(int)

# Observable numeric features only
feature_cols = [
    "search_volume",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

X = df[feature_cols].copy()
y = df["target"].copy()

# Replace missing/infinite values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
print("\nDeclining rate:", round(y.mean(), 3))

Feature matrix shape: (30000, 26)
Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.542


In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Keep all pages from the same client in only one split
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nOverlapping clients:",
      len(set(groups_train) & set(groups_test)))

print("\nTraining declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))

Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Overlapping clients: 0

Training declining rate: 0.55
Test declining rate: 0.511


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score
)

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Test predictions
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

# Model metrics
rf_roc_auc = roc_auc_score(y_test, rf_prob)
rf_ap = average_precision_score(y_test, rf_prob)
rf_precision = precision_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("ROC AUC:", round(rf_roc_auc, 3))
print("Average Precision:", round(rf_ap, 3))
print("Precision:", round(rf_precision, 3))

Random Forest Results
---------------------
ROC AUC: 0.91
Average Precision: 0.919
Precision: 0.761


In [18]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 10 features:")
display(importance.head(10))

Top 10 features:


,feature,importance
16,impressions_prev_30d,0.285285
13,impressions_last_30d,0.170180
4,impressions_90d,0.082461
19,content_age_days,0.075922
11,days_with_impressions,0.074690
22,avg_position,0.070993
14,clicks_last_30d,0.031755
2,word_count,0.025775
3,char_count,0.024234
15,sessions_last_30d,0.021964


In [19]:
# Week-4 baseline score
baseline_score = (
    (df.loc[test_idx, "content_age_days"] / 365) * 0.5
    + (df.loc[test_idx, "impressions_90d"] /
       df.loc[test_idx, "impressions_90d"].max()) * 0.5
)

# Higher score = more likely to be reviewed
baseline_roc_auc = roc_auc_score(y_test, baseline_score)
baseline_ap = average_precision_score(y_test, baseline_score)

# Precision among top 50 baseline-ranked pages
top50_idx = np.argsort(baseline_score.values)[-50:]
baseline_precision50 = y_test.iloc[top50_idx].mean()

# Precision among top 50 Random Forest predictions
rf_top50_idx = np.argsort(rf_prob)[-50:]
rf_precision50 = y_test.iloc[rf_top50_idx].mean()

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Random Forest"],
    "ROC AUC": [baseline_roc_auc, rf_roc_auc],
    "Average Precision": [baseline_ap, rf_ap],
    "Precision@50": [baseline_precision50, rf_precision50]
})

display(comparison.round(3))

,Method,ROC AUC,Average Precision,Precision@50
0,Week-4 Baseline,0.448,0.469,0.38
1,Random Forest,0.910,0.919,1.00


## 4. Errors and interpretation

The Random Forest performed better than the Week-4 baseline on the same client-grouped test split and metrics. The measured ROC AUC increased from 0.448 for the baseline to 0.910 for the Random Forest, while Precision@50 increased from 0.38 to 1.00.

The model relies most strongly on recent and previous 30-day impressions, followed by 90-day impressions, content age, days with impressions, and average position.

There were 883 false positives and 338 false negatives in the test set. The false positives show that the model can flag stable or improving pages when their observable signals resemble declining pages. The false negatives show that some genuinely declining pages can still receive a lower model probability.

For example, some false negatives had declining trends but relatively strong recent impressions or good average positions, which may make them harder for the model to distinguish from healthy pages.

These errors show that the model is not a guarantee of content problems or refresh success. The model should be used as decision-support to prioritize human review, not as an automatic refresh decision.

In [20]:
# Identify the model's errors on the test set

error_analysis = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "content_age_days",
        "impressions_90d",
        "impressions_last_30d",
        "impressions_prev_30d",
        "avg_position",
        "trend_direction"
    ]
].copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = rf_pred
error_analysis["model_probability"] = rf_prob

# False positives: predicted declining, actually not declining
false_positives = error_analysis[
    (error_analysis["predicted"] == 1) &
    (error_analysis["actual"] == 0)
]

# False negatives: predicted not declining, actually declining
false_negatives = error_analysis[
    (error_analysis["predicted"] == 0) &
    (error_analysis["actual"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nSample false positives:")
display(false_positives.head(5))

print("\nSample false negatives:")
display(false_negatives.head(5))


False positives: 883
False negatives: 338

Sample false positives:


,content_id,client_id,content_age_days,impressions_90d,impressions_last_30d,impressions_prev_30d,avg_position,trend_direction,actual,predicted,model_probability
13,content_a5a2fbc76336,client_8527a891e2,238,307,85,77,39.8,stable,0,1,0.674655
26,content_72c5c2d73e5a,client_4e07408562,300,2426,743,818,30.0,stable,0,1,0.647071
36,content_bce275871a25,client_f369cb89fc,187,371,77,95,5.4,stable,0,1,0.683550
64,content_685de0e3b7cb,client_f369cb89fc,106,2639,363,250,7.2,up,0,1,0.534656
96,content_600405887c39,client_4e07408562,545,1197,243,267,21.4,stable,0,1,0.533274



Sample false negatives:


,content_id,client_id,content_age_days,impressions_90d,impressions_last_30d,impressions_prev_30d,avg_position,trend_direction,actual,predicted,model_probability
129,content_b4170c25efd2,client_4e07408562,421,2159,403,663,21.3,down,1,0,0.478427
132,content_cfbb81c693df,client_e629fa6598,460,4540,322,426,13.7,down,1,0,0.498733
148,content_033581b09704,client_4e07408562,504,41650,5250,7594,4.4,down,1,0,0.490293
165,content_eaea09d6891e,client_4e07408562,545,1035,286,442,23.6,down,1,0,0.440731
252,content_aba4b4460e47,client_e629fa6598,460,1334,93,185,50.1,down,1,0,0.324596


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.